In [0]:
import dlt
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
looktables_rules = {
    "rule1" : "show_id is NOT NULL"
}

In [0]:
@dlt.table(
    name = "gold_netflixdirectors"
)

@dlt.expect_all_or_drop(looktables_rules)
def myfunc():
    df = spark.readStream.format("delta").load ("abfss://silver@netflixdatalake.dfs.core.windows.net/netflix_directors")
    return df

In [0]:
@dlt.table(
    name = "gold_netflixcast"
)
@dlt.expect_all_or_drop(looktables_rules)
def myfunc():
    df = spark.readStream.format("delta").load("abfss://silver@netflixdatalake.dfs.core.windows.net/netflix_cast")
    return df

In [0]:
@dlt.table(
    name = "gold_netflixcountries"
)
@dlt.expect_all_or_drop(looktables_rules)
def myfunc():
    df = spark.readStream.format("delta").load("abfss://silver@netflixdatalake.dfs.core.windows.net/netflix_countries")
    return df

In [0]:
@dlt.table(
    name = "gold_netflixcategory"
)
@dlt.expect_or_drop("rule1" , "show_id is NOT NULL")
def myfunc():
    df = spark.readStream.format("delta").load("abfss://silver@netflixdatalake.dfs.core.windows.net/netflix_category")
    return df

In [0]:
import dlt

@dlt.table(
    name="silver_netflix_title",
    comment="Silver Netflix titles loaded from ADLS"
)
def silver_netflix_title():
    return (
        spark.read   
        .format("delta")
        .load("abfss://silver@netflixdatalake.dfs.core.windows.net/netflix_titles")
    )


In [0]:
def gold_trns_netflixtitles():
    df = spark.readStream.table("LIVE.gold_stg_netflixtitle")
    df = df.withColumn("newflag", lit(1))
    return df

In [0]:
masterdata_rules = {
    "rule1" : "newflag is NOT NULL",
    "rule2" : "show_id is NOT NULL"
}

In [0]:
from pyspark.sql.functions import lit

@dlt.table(name="gold_trns_netflixtitles")
def gold_trns_netflixtitles():
    return (
        dlt.read_stream("silver_netflix_title")
           .withColumn("newflag", lit(1))
    )


In [0]:
dlt.create_streaming_live_table(
    name="gold_netflix_titles_scd2",
    comment="SCD Type-2 Netflix titles"
)

dlt.apply_changes(
    target="gold_netflix_titles_scd2",
    source="gold_trns_netflixtitles",
    keys=["show_id"],
    sequence_by="release_year",
    stored_as_scd_type=2
)
